# whisper LoRA 파인튜닝 — 608 구음장애 (large-v3-turbo)

**11차 회차로 맞춰져 있다. 위에서 아래로 전부 실행하면 된다 — 고칠 곳 없음.**

런타임: **T4 GPU**. 드라이브에 `colab_trainset_big7.zip`이 있어야 한다(재업로드 불필요).

**11차는 8·9·10차와 다르다 — 레시피가 아니라 데이터가 변수다.** VS01 단어
264화자분을 병합한 새 데이터셋(big7)의 **첫 실행**이라, 스텝 수는 셀 8이
데이터를 로드한 뒤 자동 계산하고(정확히 1에폭), dev·test baseline도 새로
잰다 — big6의 옛 수치(dev CER 0.3101 등)는 이 데이터셋에 안 맞는다.

| | 걸리는 일 |
|---|---|
| 셀 2 설치 | numpy가 안 바뀌면 **재시작 없이** 계속 간다. 바뀌면 셀이 멈추고 알려준다 |
| 셀 8 학습 | 약 1300스텝(train 10485개 기준 1에폭) · 4번 평가. 7차재현(1144스텝)보다 길다 |
| 셀 10 | 기본으로 건너뛴다(모델 크기 비교는 이미 끝났다) |

**볼 지점**
- 셀 4 → `[11차] LoRA=wide ...` / `어텐션 구현: sdpa`
- 셀 8 → `MAX_STEPS 자동계산: train 10485개 / ... = N스텝/에폭` — N이 여기 적힌
  1300 근처가 아니면 zip이 big7이 아니거나 필터링 결과가 다른 것이다
- 셀 8 → `파인튜닝 전 baseline (dev): ...` — 이 데이터셋의 새 기준선이다.
  **7차재현의 0.3101과 비교하지 마라** — 다른 시험지다
- 셀 9 dev 곡선 — 판정은 **11차 자신의 학습 전 → 후**로 한다(다른 회차와
  가로 비교 금지, 데이터가 다르므로)

학습 전 baseline을 이 회차에서 새로 잰다(위 표 「걸리는 일」 참고). LoRA는
`lora_B`가 0 초기화라 학습 전 출력이 베이스 모델과 동일 — **회차 무관**이지만
**데이터셋 무관은 아니다**(같은 베이스 모델도 다른 시험지에 다른 점수를 낸다).

## 셀 1 — GPU 확인 (Tesla T4가 떠야 함)

In [ ]:
!nvidia-smi

## 셀 2 — 설치 (환경 핀). numpy가 바뀔 때만 재시작을 요구한다

In [ ]:
# ══ 환경 고정 ═══════════════════════════════════════════════
# 회차 간 비교를 하려면 프레임워크가 회차마다 바뀌면 안 된다. 7차(8/18)는
# transformers 4.x, 7차재현(8/20)은 5.15.1이었고 그 사이에 같은 배치 설정이
# T4에서 OOM으로 죽었다. 올릴 때는 회차를 새로 시작하고 대장에 환경을 적는다.
TFV = "5.15.1"

# numpy는 Colab 이미지가 든 버전을 그대로 고정한다. 숫자를 하드코딩하면
# 이미지가 바뀔 때마다 다시 깨진다:
#   너무 높음 → AttributeError: module 'numpy' has no attribute '_no_nep50_warning'
#   너무 낮음 → ImportError: cannot import name '_center' from numpy._core.umath
import numpy, subprocess, sys
NPV = numpy.__version__
print("Colab 기본 numpy:", NPV, "→ 이 버전으로 고정 / transformers →", TFV)

# 핵심: numpy 핀을 같은 pip 호출 안에 넣어야 resolver가 도로 올리지 못한다.
!pip install -q -U "transformers=={TFV}" datasets peft accelerate evaluate jiwer librosa "numpy=={NPV}"
!pip uninstall -y -q torchao   # peft 버전검사와 충돌, LoRA엔 불필요

# **새 프로세스로** 확인한다. 이 커널이 이미 들고 있는 모듈은 pip이 디스크를
# 바꿔도 안 바뀌므로, 커널 안에서 import해 확인하면 거짓 통과한다.
_out = subprocess.run(
    [sys.executable, "-c",
     "import numpy, transformers; print(numpy.__version__, transformers.__version__)"],
    capture_output=True, text=True)
print("디스크:", (_out.stdout or _out.stderr).strip())
_disk = _out.stdout.split()

if len(_disk) != 2 or _disk[1] != TFV:
    raise SystemExit(f"transformers가 {TFV}가 아니다. 위 pip 출력을 확인해라.")
# numpy만이 재시작을 강제한다 — 이 커널이 이미 로드했기 때문이다.
# transformers·datasets 등은 아직 import 전이라 새 파일이 그대로 읽힌다.
if _disk[0] != NPV:
    raise SystemExit(f"numpy가 {NPV} → {_disk[0]}로 바뀌었다."
                     " [런타임 → 세션 다시 시작] 후 셀 2.5부터 다시 실행해라.")
print("✅ 설치 완료 — 재시작 없이 셀 2.5로 계속 진행")


## 셀 2.5 — 재시작 후 검증 (여기서 통과해야 아래가 의미 있음)
버전 3종을 찍고 실제 임포트까지 해본다. 실패하면 아래 셀들은 전부 같은 이유로 터지니 여기서 잡는다.
- `numpy has no attribute '_no_nep50_warning'` → 런타임 torch가 지금 numpy(≥2.3)를 못 따라감
- `cannot import name '_center' from numpy._core.umath` → 반대로 numpy가 너무 낮음

**처방은 둘 다 동일**: 런타임 → **세션 삭제 후 다시 시작** → 셀 2부터 다시(셀 2가 이미지 기준 numpy를 유지한다).
버전 숫자를 손으로 고쳐 맞추려 하지 말 것.

In [ ]:
import numpy, torch, transformers
print("numpy       :", numpy.__version__)
print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)
if not transformers.__version__.startswith("5.15"):
    raise SystemExit("transformers 핀이 안 먹었다 — 셀 2를 다시 실행해라.")

# torch↔numpy 불일치면 바로 아래 줄에서 터진다(셀 4의 그 에러와 같은 원인)
from transformers import WhisperForConditionalGeneration
import librosa; librosa.stft(numpy.zeros(2048, dtype=numpy.float32))   # _center 계열 조기 탐지
print("✅ 임포트 정상 — 셀 3으로")

## 셀 3 — 드라이브 마운트 + 업로드한 zip 풀기 + 경로 확인
`ZIP`을 본인이 올린 위치로 수정.

In [ ]:
import os, torch, json
from google.colab import drive
drive.mount('/content/drive')

ZIP  = "/content/drive/MyDrive/colab_trainset_big7.zip"   # ← 업로드한 zip 경로
# big7 = VS01 병합본(11차부터). train 2281→10485, dev 326→1427, test 417→2673 —
# **train만 4.6배가 아니라 dev·test도 커졌다.** 그래서 옛 dev/test 수치(0.3101 등)를
# 그대로 인용하면 안 된다 — 시험지 자체가 다른 시험지다(아래 셀 8 참고).
BASE = "/content/608-trainset"                       # 풀 위치(세션 로컬, 빠름)
if os.path.exists(ZIP):
    !unzip -oq "$ZIP" -d "$BASE"
else:
    # zip 대신 폴더째 올렸으면 그 경로를 BASE로 지정하고 위 unzip은 건너뛴다
    BASE = "/content/drive/MyDrive/608-trainset"

print("CUDA:", torch.cuda.is_available())

# dev는 선택이 아니다. dev 없이 돌리면 test로 체크포인트를 고르게 되고, 그러면
# 보고되는 CER이 홀드아웃 성능이 아니라 "시험지를 보며 고른 점수"가 된다.
# 4~6차 배치가 그 상태였다. 없으면 여기서 멈추고 재패키징을 요구한다.
if not os.path.exists(os.path.join(BASE, "dev.jsonl")):
    raise SystemExit(
        "dev.jsonl이 없다 — 이 zip은 dev 스플릿 없이 만들어졌다.\n"
        "prepare_colab_trainset.py를 --dev-speaker-frac 0.1 로 다시 돌려 패키징해라.\n"
        "(dev 없이 진행하면 test로 최고 체크포인트를 고르게 되어 수치를 못 쓴다)"
    )

for f in ["train.jsonl","dev.jsonl","test.jsonl"]:
    p=os.path.join(BASE,f); n=sum(1 for _ in open(p,encoding="utf-8"))
    print(f"  {f}: {n}개")
print("wav:", sum(len(fs) for _,_,fs in os.walk(os.path.join(BASE,'wav'))), "개")

## 셀 4 — 모델 + LoRA (+ 회차 설정: RUN)

In [ ]:
# 할당자가 크기별로 굳은 세그먼트를 재사용하지 못해 여유가 있는데도 OOM이 나는
# 경우가 있다(생성 평가는 배치마다 시퀀스 길이가 달라 특히 그렇다). 이 설정은
# **CUDA 컨텍스트가 만들어지기 전에** 걸려야 효과가 있어 맨 위에 둔다.
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import torch
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from peft import LoraConfig, get_peft_model

# 베이스 모델. zero-shot 비교(셀 10) 결과로 large-v3-turbo를 채택했다.
#
#   모델              단어 CER   문장 CER
#   small               2.000      0.404
#   medium              1.376      0.363
#   large-v3-turbo      0.890      0.321   ← 두 지표 모두 승
#
# 단어가 모델 크기에 훨씬 민감하다(2.2배 개선 vs 문장 1.26배). 앱 최약점이
# 단어 수준 구음장애라 이 차이가 그대로 제품 성능이다. turbo는 디코더가 얕아
# large-v3(1550M)보다 훨씬 빠르면서 T4에 들어간다.
#
# 되돌리려면 이 한 줄만 바꾼다. 프로세서도 같은 ID로 로드되므로 mel 빈 수
# (large-v3 계열 128 vs small/medium 80)가 자동으로 맞는다 — 프로세서를
# 재사용하면 입력이 조용히 깨진다.
MODEL_ID = "openai/whisper-large-v3-turbo"

processor = WhisperProcessor.from_pretrained(MODEL_ID, language="ko", task="transcribe")
# **fp32로 명시 로드해야 한다.** large-v3-turbo는 config에 torch_dtype: float16이
# 박혀 있어 최신 transformers가 fp16 가중치로 올린다. 그러면 특징 추출기가 내놓는
# fp32 입력과 어긋나 인코더 conv에서 터진다:
#   RuntimeError: Input type (float) and bias type (c10::Half) should be the same
# small은 config가 fp32라 이 문제가 없어 지금까지 안 드러났다.
#
# 혼합정밀은 Trainer의 fp16=True(autocast)가 맡는다. 가중치까지 fp16으로 두고
# 학습하면 마스터 가중치가 없어 불안정하다. T4는 Turing이라 bf16이 없어 이 조합이
# 유일한 선택지다.
# **attn_implementation을 명시한다.** 기본값에 맡기면 transformers 버전에 따라
# eager로 떨어질 수 있고, 그러면 어텐션 확률 행렬이 통째로 저장된다:
#   배치4 x 헤드20 x 1500 x 1500 x 2B = 360MB/층  x  인코더 32층 = 11.5GB
# sdpa는 그걸 아예 만들지 않는다. 이 한 줄이 T4에 들어가느냐 마느냐를 가른다.
# (셀 2가 transformers를 핀 없이 -U로 깔기 때문에 기본값에 기댈 수 없다.)
try:
    model = WhisperForConditionalGeneration.from_pretrained(
        MODEL_ID, torch_dtype=torch.float32, attn_implementation='sdpa')
except (ValueError, ImportError) as e:
    print('sdpa 거부됨:', e)
    model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID, torch_dtype=torch.float32)
print('어텐션 구현:', getattr(model.config, '_attn_implementation', '?'),  # 'sdpa'여야 한다
      '| torch', torch.__version__)
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
model.config.use_cache = False

# ══ 8차 · 9차 실험 설계 ═══════════════════════════════════════
# 회차마다 **변수는 하나**다. 아래 _CONF 표가 그 규율을 구조로 강제한다.
#
#   7차(채택본)  넓게 · x1   → dev CER 0.1337
#     8차        좁게 · x1   → 변수: LoRA 폭      (7차와 비교)
#     9차        좁게 · x3   → 변수: 단어 비중    (8차와 비교)
#
# ── 8차: LoRA 폭 ──────────────────────────────────────────────
# 7차는 두 가지를 동시에 바꿨다: ① 데이터를 전 세그먼트 재정렬본으로 교체
# ② LoRA 대상을 q/v → 어텐션 전체+FFN으로 확장(#43). 결과는 좋았지만(단어 80.1%)
# **어느 쪽이 기여했는지 모른다.** 8차는 그걸 가른다.
#
#   좁게: ["q_proj", "v_proj"]                                  ← 8차 A조건(현재)
#   넓게: ["q_proj","k_proj","v_proj","out_proj","fc1","fc2"]   ← 7차가 쓴 값
#
# **데이터·zip·에폭·시드·생성설정은 7차와 전부 고정한다.** 변수는 이 줄 하나다.
#
# ── 판정 규칙 (실행 전에 못 박는다) ──────────────────────────────
# 우열은 **dev로** 가린다. test로 가리면 7차 이전의 실수(시험지 보고 답 고르기)를
# 그대로 반복한다. dev로 승자를 정하고, 승자만 test를 한 번 본다.
#
#   8차 dev CER  vs  7차 dev CER 0.1337
#     차이 ≥ 0.01  → 좋은 쪽 채택
#     차이 < 0.01  → **차이 없음으로 읽고 좁은 쪽 채택**
#
# 0.01 기준의 근거: 7차 곡선에서 3에폭(0.1337)과 4에폭(0.1341)의 차이가 0.0005였다.
# 그 정도는 노이즈다. 조건당 1회 실행이라 시드 분산을 못 재므로, 작은 차이를
# 실력 차로 읽으면 안 된다. 동률이면 좁은 쪽 — 넓은 쪽은 학습 파라미터만 더 쓴다.
# 이 기준을 결과를 본 뒤에 정하면 유리한 쪽으로 해석하게 된다.
#
# Whisper의 어텐션 출력층 이름은 `out_proj`다(`o_proj` 아님 - Llama 계열과 다르다).
# FFN은 `fc1`/`fc2`다. 이름이 틀리면 PEFT가 그 모듈만 조용히 건너뛰어, "넓혔는데
# 왜 파라미터가 안 늘지"로 헤매게 된다. 그래서 붙이기 전에 실재 여부를 확인한다.
# ── 9차: 단어 업샘플링 ────────────────────────────────────────
# train 세그먼트는 단어 1209 · 문장 1072로 반반인데, 단어 클립이 0.6초라
# **시간 비중은 11.4% : 88.6%** 로 기운다(단어 12.2분 vs 문장 94.9분).
# 앱 병목은 단어인데(단어 CER 0.276 vs 문장 0.135) 학습 예산의 1/9만 쓰는 셈이다.
# 9차는 단어 행을 3배로 반복해 시간 비중을 28%로 올린다. 새 데이터는 없다.
#
#   판정: **dev 단어 CER**이 주 지표, **dev 문장 CER**이 대가(cost) 지표.
#   문장이 얼마나 나빠지는지를 보고 트레이드오프를 판단한다.
#
#   ⚠️ 9차의 비교 대상은 7차가 아니라 **8차**다. LoRA 폭을 8차와 같게 두어야
#   변수가 업샘플 하나로 남는다. 8차를 먼저 돌려야 비교 대상이 생긴다.
#   8차가 "넓은 쪽 채택"으로 끝나면 아래 9차 줄도 "wide"로 바꾸고, 그때는
#   7차에 과제별 dev 수치가 없으므로 "7차재현"을 한 번 돌려야 한다.
#
#   ⚠️ 업샘플은 스텝 수도 늘린다(train 2281 → 4699, 에폭당 2.06배). 이건 혼입이
#   아니라 개입의 일부다 — "단어에 학습 예산을 더 쓴다"가 곧 이 실험이다.
#   과학습은 dev 기반 최고 체크포인트 복원이 흡수한다.

# ══ 회차 설정 — 여기 한 줄만 바꾼다 ═══════════════════════════
RUN = "11차"        # "7차재현" | "8차" | "9차" | "10차" | "11차"   ← 회차는 여기 한 줄

_WIDE   = ["q_proj", "k_proj", "v_proj", "out_proj", "fc1", "fc2"]
_NARROW = ["q_proj", "v_proj"]
_CONF = {          # 회차 → (LoRA 폭, 단어 업샘플 배수, 스텝)   비교 대상
    "7차재현": ("wide",   1, 1144),  # **먼저 돌린다.** 아래 세 가지를 한 번에 한다:
                                    #  ① 노트북 최적화가 수치를 안 바꿨는지 검증(0.1337 재현)
                                    #  ② 7차엔 없는 dev 단어/문장 CER 확보 → 9차의 비교 대상
                                    #  ③ test baseline(회차 무관 상수)을 여기서 한 번 확정
    "8차":     ("narrow", 1, 1144),  # ← 7차와 비교 (변수: LoRA 폭).  졌다
    "9차":     ("wide",   3, 1144),  # ← 7차재현과 비교 (변수: 단어 비중). 졌다
    "10차":    ("wide",   1, 2288),  # ← 7차재현과 비교 (변수: **학습량**)
    #   ── 10차를 학습량으로 잡은 이유 ──────────────────────────
    #   값싼 손잡이 둘이 연달아 기각됐다. 8차(LoRA 폭)는 좁은 쪽이 과소적합으로
    #   졌고, 9차(단어 업샘플 x3)는 7차재현과 4자리까지 동률이었다.
    #   남은 손잡이 중 **방향 신호가 있는 건 학습량 하나뿐**이다:
    #   7차재현도 8차도 1144에서 dev CER이 아직 내려가는 중이었다
    #   (재현 858→1144에서 −0.0032, 8차 −0.0051). 곡선이 안 꺾였는데 멈춘 것이다.
    #
    #   그리고 이건 이 노트북에 자동 이어붙이기가 생겨서 **비로소 가능해진**
    #   실험이다. 무료 Colab 세션이 버티는 양은 700~850스텝이라 2288은 한 세션에
    #   절대 안 끝난다. 예전 같으면 매번 0부터라 영원히 못 끝냈다.
    #
    #   ⚠️ **판정은 전체·문장 CER로만 한다.** dev 단어 CER은 재현 오차가 0.037이라
    #   (같은 설정·같은 시드 두 번의 실측) 이 크기의 차이를 구분하지 못한다.
    #   단어로 판정하려면 dev 단어 세트를 먼저 키워야 하고, 그건 학습 데이터를
    #   건드려 비교 기준을 리셋하므로 11차 몫이다.
    #
    #   ⚠️ **스텝별 수치를 재현과 나란히 놓지 마라.** LR이 max_steps에 걸쳐
    #   선형 감쇠하므로 2288로 늘리면 **모든 스텝의 LR이 달라진다.** 10차의
    #   286스텝은 재현의 286스텝과 같은 조건이 아니다. 비교는 **최고 대 최고**로 한다.
    "11차":    ("wide", 1, "auto"),  # ← **레시피가 아니라 데이터**가 변수다
    #   ── 11차가 앞의 셋과 다른 이유 ────────────────────────────
    #   8·9·10차는 전부 같은 데이터(big6) 위에서 레시피만 바꿨고 전부 문턱을
    #   못 넘었다. 세 번 연속 같은 답이면 "레시피가 아니라 데이터"라는 뜻이다
    #   (대장 9차 절 참고). 11차는 처음으로 **데이터를 바꾼다** — VS01 단어
    #   264화자분을 병합한 big7(train 2281→10485, 단어 화자 26→112명).
    #
    #   레시피는 7차재현과 동일하게 고정한다(넓은 LoRA·x1·같은 LR/배치/시드).
    #   그래야 "데이터를 늘리니 좋아졌나"에 순수하게 답한다 — 레시피까지
    #   같이 바꾸면 8·9·10차의 실수(변수 둘)를 반복하는 것이다.
    #
    #   ⚠️ **스텝을 "auto"로 둔다 — 1144를 그대로 못 쓴다.** train이 4.6배가
    #   됐는데 스텝을 고정하면 1144스텝은 1에폭도 안 된다(2281/8=286스텝/에폭
    #   이던 게 10485/8=1311스텝/에폭이 된다). "auto"는 셀 8이 데이터셋을
    #   로드한 뒤 **정확히 1에폭**(모든 학습 예제를 최소 한 번은 보게)으로
    #   스텝 수를 계산한다.
    #
    #   ⚠️ **dev·test baseline을 다시 잰다.** big7의 dev(1427개)·test(2673개)는
    #   big6과 **다른 시험지**다 — 옛 수치(dev 0.3101, test 단어 51.4% 등)를
    #   그대로 인용하면 안 된다. 셀 8에서 11차도 baseline을 재도록 했다.
    #
    #   ⚠️ **7차재현과 직접 비교하지 마라.** 데이터가 다르므로 dev CER이
    #   좋아져도 "레시피가 나아서"인지 "시험지가 쉬워져서"인지 못 가른다.
    #   11차의 진짜 비교 대상은 **11차 자신의 학습 전 baseline**이다
    #   (같은 실행의 전→후만 유효하다는 이 프로젝트의 원칙 그대로).
    #   레시피 비교가 다시 필요해지면 그건 12차 이후, big7 위에서 한다.
}
_width, WORD_UPSAMPLE, RUN_STEPS = _CONF[RUN]
LORA_TARGETS = _NARROW if _width == "narrow" else _WIDE
print(f"[{RUN}] LoRA={_width} {LORA_TARGETS} | 단어 업샘플 x{WORD_UPSAMPLE} | {RUN_STEPS}스텝")

import torch.nn as nn
present = {n.rsplit(".", 1)[-1] for n, m in model.named_modules() if isinstance(m, nn.Linear)}
missing = [t for t in LORA_TARGETS if t not in present]
if missing:
    print("모델에 없는 LoRA 대상:", missing)
    print("이 모델의 Linear 층 이름:", sorted(present))
    raise SystemExit("LORA_TARGETS를 위 목록에 맞게 고쳐라.")
print("LoRA 대상 확인:", LORA_TARGETS)

model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, target_modules=LORA_TARGETS, lora_dropout=0.05))
model.print_trainable_parameters()
# 7차(넓게)의 학습 파라미터 수와 대조해 조건이 실제로 바뀌었는지 확인한다.
# 이 숫자가 7차와 같으면 LORA_TARGETS 교체가 반영되지 않은 것이다 —
# 드라이브에 있던 예전 노트북 사본을 돌리면 실제로 이렇게 된다(5차에서 겪었다).

# 생성(평가) 설정 — 이거 없으면 whisper가 반복 환각으로 CER이 2.0+로 부풀려진다.
model.generation_config.language = "ko"
model.generation_config.task = "transcribe"
model.generation_config.no_repeat_ngram_size = 3   # 같은 구절 반복 억제

# ── 평가 시간의 대부분은 여기서 결정된다 ─────────────────────────
# 자기회귀 생성이라 **배치 안에서 가장 긴 시퀀스**가 그 배치의 시간을 정한다.
# 그리고 학습 전/초기 모델은 환각으로 거의 항상 상한까지 뱉는다 — 즉 상한값이
# 곧 평가 비용이다.
#
# 실측 라벨 길이 (colab_trainset_big6):
#     train 단어  중앙 1자 · 최대  2자
#     train 문장  중앙 13자 · p95 34자 · 최대 93자
#     dev  문장  최대 56자   |   test 문장 최대 44자
#
# 100토큰은 dev/test 최장(56자)의 두 배 가까이라 낭비다. 72로 낮춰 환각 꼬리를
# 자른다. 잘림 위험은 없다 — 한국어는 음절당 1토큰 안팎이라 56자면 60토큰 미만이다.
model.generation_config.max_new_tokens = 72

# **KV 캐시를 반드시 켠다.** 위에서 config.use_cache=False로 둔 건 학습용 관례인데,
# 그게 생성까지 따라가면 매 토큰마다 전체 시퀀스를 재계산해 평가가 몇 배 느려진다.
# (gradient_checkpointing을 안 쓰므로 학습에도 use_cache=False가 필요 없지만,
#  건드리면 학습 조건이 바뀌므로 생성 쪽만 명시적으로 켠다.)
model.generation_config.use_cache = True


## 셀 5 — 데이터셋 로드 + 전처리 (+ 단어 업샘플링)

In [ ]:
from datasets import load_dataset, Audio, concatenate_datasets
# 세 갈래다. dev는 학습 중 고르기용(에포크 평가·최고 체크포인트), test는 학습이
# 끝난 뒤 딱 한 번만 본다. 이 분리가 깨지면 보고 수치가 홀드아웃이 아니게 된다.
ds = load_dataset("json", data_files={
    "train": f"{BASE}/train.jsonl",
    "dev":   f"{BASE}/dev.jsonl",
    "test":  f"{BASE}/test.jsonl"})
ds = ds.map(lambda b: {"audio": os.path.join(BASE, b["audio"])})
ds = ds.cast_column("audio", Audio(sampling_rate=16000))

if "WORD_UPSAMPLE" not in globals():
    raise SystemExit("셀 4를 먼저 실행해라 — RUN/WORD_UPSAMPLE이 거기서 정해진다.")

# task_type(단어/문장)은 아래 prep 매핑에서 사라지므로 먼저 빼 둔다.
# 구 배치 zip은 이 필드가 없을 수 있어, 없으면 None으로 두고 셀 9에서 건너뛴다.
def _tt(split):
    return list(ds[split]["task_type"]) if "task_type" in ds[split].column_names else None
dev_task_types  = _tt("dev")
test_task_types = _tt("test")
train_word_idx  = [i for i, t in enumerate(_tt("train") or []) if t == "wordlist"]

# ── RAM 방어 — big7부터 예제 수가 4.7배(3124→14773)로 늘었다 ─────────────
# large-v3-turbo는 mel 128빈이라 예제당 input_features가 128x3000x4바이트 =
# 약 1.46MB다(small/medium은 80빈이라 이 절반 이하). datasets.map()의 기본
# writer_batch_size(1000)로 그대로 돌리면 배치 하나가 1.46GB를 RAM에 버퍼링
# 하고, train만 10485개면 이론상 15GB까지 쌓일 수 있다 — 무료 Colab의 시스템
# RAM(약 12GB)을 넘긴다. big6(3124개, 4.4GB치)까지는 안 드러나던 문제다.
#
# writer_batch_size를 낮추면 **결과는 완전히 같다**(디스크에 쓰는 단위만
# 작아진다) — 순수 메모리 대 시간 트레이드오프다.
print("[RAM] map 시작 전:")
!free -h | head -2

def prep(b):
    a = b["audio"]
    b["input_features"] = processor.feature_extractor(
        a["array"], sampling_rate=16000).input_features[0]
    b["labels"] = processor.tokenizer(b["text"]).input_ids
    return b
ds = ds.map(prep, remove_columns=ds["train"].column_names, writer_batch_size=20)

import gc
gc.collect()
print("[RAM] map 종료 후:")
!free -h | head -2

# ── 단어 업샘플링 (train 전용, **전처리 뒤**) ──────────────────────
# 순서가 중요하다. 전처리 앞에서 복제하면 **같은 오디오의 mel을 배수만큼 다시
# 계산한다**(x3이면 2418개분 낭비, CPU 몇 분). select()는 인덱스만 복제하므로
# 추가 계산이 0이다.
#
# **dev/test는 절대 건드리지 않는다.** 잣대가 바뀌면 8차와 비교할 수 없다.
if WORD_UPSAMPLE > 1:
    if not train_word_idx:
        raise SystemExit("train.jsonl에 task_type이 없다 — 구 배치 zip이다. big6 이상을 써라.")
    _before = len(ds["train"])
    ds["train"] = concatenate_datasets(
        [ds["train"]] + [ds["train"].select(train_word_idx)] * (WORD_UPSAMPLE - 1))
    _n_word = len(train_word_idx) * WORD_UPSAMPLE
    print(f"단어 업샘플 x{WORD_UPSAMPLE}: train {_before} -> {len(ds['train'])}"
          f" (단어 {_n_word} · 문장 {len(ds['train']) - _n_word})")
print(ds)

## 셀 7.5 — 앱 채점기 로드 (제품과 같은 잣대)

608 평가의 표준 CER은 **앱 조건보다 비관적**이다. 앱은 단어 과제에 음소 유사 가중
거리를 써서 조음 유사 혼동(ㅂ↔ㅍ, 종성 탈락)을 부분 오류로 본다. 같은 채점기로
재야 "이 모델을 붙이면 앱이 얼마나 좋아지나"에 답할 수 있다.

무엇보다 앱이 내는 판정은 오류율이 아니라 **임계값 통과 여부**(≤0.34)다.
평균이 경계 아래인 것과 항목별 통과율은 다른 이야기다.


In [ ]:
# scripts/asr_eval/app_scorer.py 를 Colab으로 가져온다.
# 리포를 통째로 올리기 번거로우면 이 셀에 붙여넣어도 되지만, 원본이 바뀌면
# 어긋나므로 파일을 올리는 쪽을 권한다.
#
# 원본(단일 진실): frontend/src/memory-link/patient/quiz/domain/phoneticDistance.ts
#                  frontend/src/memory-link/patient/quiz/domain/speechScore.ts
# 파이썬 이식본: scripts/asr_eval/app_scorer.py (test_app_scorer.py가 두 구현을 고정)

import os, sys, urllib.request

APP_SCORER = "/content/app_scorer.py"
if not os.path.exists(APP_SCORER):
    # (a) 드라이브에 올려 뒀으면 그걸 쓴다
    cand = "/content/drive/MyDrive/app_scorer.py"
    if os.path.exists(cand):
        import shutil; shutil.copy(cand, APP_SCORER)
    else:
        raise SystemExit(
            "app_scorer.py가 없다. scripts/asr_eval/app_scorer.py를 드라이브"
            " 루트에 올리거나 /content/에 업로드하라."
        )
sys.path.insert(0, "/content")
import app_scorer
print("앱 채점기 로드 · 정답 임계값:", app_scorer.SPEECH_PASS_THRESHOLD)


## 셀 6 — 데이터 콜레이터 (whisper 필수)

In [ ]:
from dataclasses import dataclass
from typing import Any
@dataclass
class Collator:
    processor: Any
    def __call__(self, feats):
        batch = self.processor.feature_extractor.pad(
            [{"input_features": f["input_features"]} for f in feats], return_tensors="pt")
        labels = self.processor.tokenizer.pad(
            [{"input_ids": f["labels"]} for f in feats], return_tensors="pt")
        lab = labels["input_ids"].masked_fill(labels.attention_mask.ne(1), -100)
        if (lab[:,0] == self.processor.tokenizer.bos_token_id).all():
            lab = lab[:,1:]
        batch["labels"] = lab
        return batch
collator = Collator(processor)

## 셀 7 — CER 지표 (전체 + 단어/문장 분리)

In [ ]:
import evaluate
cer_metric = evaluate.load("cer")

# 에폭 평가에서 **단어/문장 CER을 따로** 낸다. 9차의 판정 지표가 dev 단어 CER이라
# 전체 CER만으로는 판단할 수 없다(단어가 세그먼트의 절반이지만 짧아서 전체
# CER에 묻힌다).
#
# 어느 스플릿인지는 **길이로** 가린다 — dev 326 vs test 417이라 겹치지 않는다.
# 재패키징으로 두 길이가 같아지면 이 분기가 조용히 틀리므로 아래에서 막는다.
TASK_TYPES = {}
for _name, _tt in (("dev", dev_task_types), ("test", test_task_types)):
    if not _tt:
        continue
    if len(_tt) in TASK_TYPES:
        raise SystemExit(f"dev와 test 길이가 같다({len(_tt)}) — 길이로 스플릿을 못 가른다.")
    TASK_TYPES[len(_tt)] = _tt

def compute_metrics(p):
    lab = p.label_ids
    lab[lab == -100] = processor.tokenizer.pad_token_id
    pred = processor.tokenizer.batch_decode(p.predictions, skip_special_tokens=True)
    ref  = processor.tokenizer.batch_decode(lab, skip_special_tokens=True)
    out = {"cer": cer_metric.compute(predictions=pred, references=ref)}
    tt = TASK_TYPES.get(len(ref))
    if tt:
        for _short, _key in (("word", "wordlist"), ("sent", "narrative")):
            idx = [i for i, t in enumerate(tt) if t == _key]
            if idx:
                out[f"cer_{_short}"] = cer_metric.compute(
                    predictions=[pred[i] for i in idx],
                    references=[ref[i] for i in idx])
    return out

## 셀 8 — 학습 (스텝 고정 · 계측 스위치 · 끊기면 이어서)

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

# ══ 출력 폭탄 억제 — Colab 프런트엔드가 멈추는 걸 막는다 ══════════════
# generate()가 호출될 때마다(평가 배치마다) transformers가 같은 경고
# (attention_mask · max_new_tokens/max_length · logits processor)를 반복
# 찍는다. 평가당 배치 수만큼(여기선 수십 회) 쌓이면 브라우저 쪽 렌더링이
# 그 출력량을 못 따라가 셀이 멈춘 것처럼 보이는 사고가 났다(9차, 572스텝
# 평가 직후). 커널은 살아 있는데 실행 표시만 멈추고 인터럽트도 안 먹는
# 증상이 이 폭탄과 시간이 겹쳤다. 학습 로직과는 무관하니 재현 결과에
# 영향을 주지 않는다 — 그냥 콘솔이 조용해질 뿐이다.
import warnings, transformers as _tf
_tf.logging.set_verbosity_error()
warnings.filterwarnings("ignore")
# ══ 계측 스위치 — 여기서 런타임의 절반이 정해진다 ══════════════
# 생성 기반 평가가 이 노트북의 지배적 비용이다. 회차당 생성 횟수:
#
#   dev 에폭 평가   326 x 4 = 1304   ← 판정에 필요. 유지
#   dev 학습전 baseline   326        ← LoRA는 B=0 초기화라 학습 전 출력이
#   test 학습전 baseline  417          **베이스 모델과 완전히 동일**하다.
#                                      회차와 무관한 상수라 한 번만 재면 된다.
#   dev 학습후 재평가     326        ← best_metric에 이미 있다. 순수 낭비
#   test 학습후           417        ← **채택 회차에서만**
#
# 전부 켜면 2790회, 탐색 회차에 필요한 건 1304회다. 절반 이상이 군더더기였다.
# baseline 두 개는 **데이터셋이 바뀔 때마다 한 번씩만** 켠다.
#
#   big6 위에서(7차재현이 첫 회차): 2026-08-20에 이미 쟀고 대장에 박았다.
#     dev  val loss 4.534758 · CER 0.3100597 (단어 1.2664 / 문장 0.1893)
#     test 단어 정확도 51.4% · 단어 CER 0.895 · 문장 CER 0.191
#          앱 통과율 검사 59.6% / 대화 66.4%
#     8·9·10차가 이 위에서 돌았으므로 계속 생략한다.
#
#   big7 위에서(11차가 첫 회차): dev 1427개·test 2673개로 **다른 시험지**다.
#     big6의 위 숫자를 그대로 쓰면 안 된다 — 여기서 새로 잰다.
#
# LoRA가 B=0 초기화라 baseline은 "회차 무관"은 맞지만 "데이터셋 무관"은
# 아니다 — 같은 베이스 모델도 다른 시험지에 대면 다른 점수가 나온다.
FIRST_RUN_BIG6 = "7차재현"
FIRST_RUN_BIG7 = "11차"
MEASURE_DEV_BASELINE = RUN in (FIRST_RUN_BIG6, FIRST_RUN_BIG7)
MEASURE_TEST_BEFORE  = RUN in (FIRST_RUN_BIG6, FIRST_RUN_BIG7)
MEASURE_TEST_AFTER   = False            # 채택이 정해진 뒤 셀 9에서 True로 바꿔 다시 실행

# ══ 학습 예산을 회차 간 고정한다 ═══════════════════════════════
# 7차: train 2281 / (배치4 x 누적2) = 286스텝/에폭 x 4에폭 = **1144 스텝**.
# 9차는 업샘플로 데이터가 2.06배라 같은 "4에폭"이 2352스텝 = 시간 2배가 된다.
# 그러면 변수가 둘(단어 비중 + 학습량)이 되고 세션이 끊길 위험도 커진다.
#
# 그래서 에폭이 아니라 **스텝으로 고정**한다. x1 회차에서는 정확히 4에폭과
# 같으므로 8차는 7차와 동일 조건이고, 9차는 "같은 예산을 어디에 쓰나"가 된다.
# 배치/누적을 이름 있는 변수로 뽑는다 — 아래 args=...와 스텝 계산 둘 다 이
# 값을 봐야 한다. 예전엔 args= 안에 리터럴로만 있어서, 스텝 자동계산을
# 넣으려니 같은 숫자(2, 4)를 두 곳에 다시 적어야 했다. 그러면 나중에 배치를
# 바꿀 때 한쪽만 고치는 사고가 난다.
BATCH_SIZE  = 2
GRAD_ACCUM  = 4   # 유효 배치 8. T4 16GB에서 4x2는 OOM — 대장 「무료 콜랩」 절 참고

# 스텝 수는 **셀 4의 회차 표**에서 온다. 여기에 숫자를 적으면 회차 설정이
# 두 곳으로 갈라져, 회차만 바꾸고 스텝을 안 바꾸는 사고가 난다.
#
# "auto"(11차)는 **정확히 1에폭**으로 해석한다 — train이 4.6배(2281→10485)가
# 됐으니 옛 고정값 1144는 1에폭에도 못 미친다(1311스텝/에폭). 스텝을 고정한
# 채로 데이터만 늘리면 새로 들어온 예제 중 일부를 아예 한 번도 안 보는
# 채로 "학습 끝"이 될 수 있다 — VS01을 받은 목적 자체가 무색해진다.
if RUN_STEPS == "auto":
    STEPS_PER_EPOCH = -(-len(ds["train"]) // (BATCH_SIZE * GRAD_ACCUM))   # 올림 — 1에폭이 안 깎이게
    MAX_STEPS = STEPS_PER_EPOCH
    # 평가점 계산은 **내림**이어야 한다. 올리면 4번째 평가점이 MAX_STEPS를
    # 넘어(1311스텝이면 올림 328 x 4 = 1312 > 1311) 실제로는 3개만 찍힌다.
    # 내림은 항상 MAX_STEPS 안에 들어온다(4 x floor(n/4) <= n이 항상 성립).
    EVAL_EVERY = max(1, STEPS_PER_EPOCH // 4)   # 4개 평가점 — 이전 회차들과 같은 관례
    print(f"[{RUN}] MAX_STEPS 자동계산: train {len(ds['train'])}개 / "
          f"(배치{BATCH_SIZE}×누적{GRAD_ACCUM}) = {STEPS_PER_EPOCH}스텝/에폭 = 1에폭"
          f" · 평가점 {EVAL_EVERY}·{EVAL_EVERY*2}·{EVAL_EVERY*3}·{EVAL_EVERY*4}")
else:
    MAX_STEPS  = RUN_STEPS   # 7차재현·8차·9차 = 1144 / 10차 = 2288
    EVAL_EVERY = 286         # x1·1144일 때 에폭 경계와 일치. 10차는 평가점이 8개가 된다

# 체크포인트를 세션 로컬(/content)에 두면 런타임이 끊길 때 같이 사라진다.
# 드라이브에 두면 재접속 후 이어서 학습할 수 있다(LoRA라 용량이 작다).
CKPT_DIR = f"/content/drive/MyDrive/whisper-608-ckpt-{RUN}"

# ══ 메모리: 왜 그래디언트 체크포인팅을 안 쓰는가 ═══════════════
# 켜 봤다. **학습이 망가진다.** 7차와 같은 지점(286스텝=1에폭)에서:
#     7차   train 1.225 · val 0.3096 · dev CER 0.1700
#     켜고  train 3.514 · val 1.0946 · dev CER 0.8695  ← baseline 0.3101보다 나쁨
#
# 같은 실행의 dev baseline은 7차와 소수점 4자리까지 일치했다(4.534758 / 0.3100597).
# **순전파와 손실이 동일한데 학습만 뒤처졌다** — 원인이 역전파 경로에만 있다는 뜻이고,
# 그 조건을 만족하는 변경은 체크포인팅뿐이었다. use_reentrant=False면 안전하다는
# 통념이 이 조합(transformers 5.15 + PEFT + Whisper)에서는 성립하지 않는다.
#
# 다시 켜려면 먼저 286스텝 dev CER이 0.17 근처로 나오는지부터 확인해라.
LOW_MEM = False


args = Seq2SeqTrainingArguments(
    output_dir=CKPT_DIR,
    # large-v3-turbo(809M)는 small(244M)보다 크다. T4 16GB에서 배치 8은 위험해
    # 4로 낮추고 누적 2로 유효 배치 8을 유지한다(학습 동역학은 그대로).
    # OOM이 나면 배치 2 · 누적 4, 그래도 안 되면 gradient_checkpointing=True.
    # 배치 2 x 누적 4 = 유효 배치 8. 7차는 4 x 2로 같은 유효 배치였는데,
    # transformers 5.15 + torch 2.11에서는 4가 T4에 안 들어간다(OOM).
    # 저장되는 활성값이 절반이 되고, transformers 5는 누적 스텝 간 손실을
    # num_items_in_batch로 정규화하므로 그래디언트는 수학적으로 같다.
    # 체크포인팅과 달리 역전파 경로를 건드리지 않는다.
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=LOW_MEM,
    # 비재진입 방식. PEFT에서 재진입(reentrant) 체크포인팅은 입력이
    # requires_grad가 아니면 그래디언트가 조용히 0이 된다 - 학습이 도는데
    # 아무것도 안 배우는 최악의 실패 방식이다. False면 그 함정이 없다.
    gradient_checkpointing_kwargs={'use_reentrant': False},
    # ⚠️ 8에서 손대지 마라. 16으로 올렸다가 학습 중 평가(286스텝)에서 OOM이 났다.
    # 학습 전 baseline은 같은 16으로 멀쩡히 통과한다 - 그때는 옵티마이저 상태도
    # 그래디언트 버퍼도 아직 없기 때문이다. 그래서 '평가는 메모리가 남는다'는
    # 추론이 틀렸다: 학습 루프 안의 평가는 학습 메모리 위에 얹힌다.
    # 8이 7차가 쓴 값이고, 재현 회차는 여기서 갈라지면 안 된다.
    per_device_eval_batch_size=8,
    dataloader_num_workers=2,
    learning_rate=1e-4, warmup_steps=20,       # LoRA whisper 표준 LR
    # 7차 곡선은 3에폭(=858스텝)에서 평탄해졌다. 1140스텝이면 그 지점을 지나며,
    # 나빠져도 아래 load_best_model_at_end가 이전 체크포인트를 복원한다.
    max_steps=MAX_STEPS, fp16=True,
    eval_strategy="steps", eval_steps=EVAL_EVERY, logging_steps=10,
    # val loss는 끝까지 내려가는데 CER은 아니라(지표 갈라짐) loss 기준으로는
    # 최고를 못 고른다. CER을 기준으로 최고 체크포인트를 복원한다.
    # 이게 없어 5차에서 0.292 대신 0.319를 저장했다(-8% 손실).
    # **이 CER은 dev의 CER이다.** test로 고르면 그 test 수치는 더 이상 홀드아웃이
    # 아니다 — 고르기에 쓴 데이터로 성능을 보고하는 셈이라 낙관적으로 편향된다.
    save_strategy="steps", save_steps=EVAL_EVERY, save_total_limit=4,   # 에폭별로 다 남긴다.
    #   10차는 평가점이 8개라 4개만 남는다. 이어붙이기에는 최신 하나면 충분하고,
    #   최고 체크포인트는 load_best_model_at_end가 따로 붙들므로 안전하다.
    #   7차재현에서 전체 CER 최선(1144)과 단어 CER 최선(858)이 갈렸다.
    #   선택 기준은 그대로 "cer"이다(회차 간 비교 가능성). 저장 개수만
    #   늘려 과제별로 어느 지점이 좋은지 사후에 볼 수 있게 한다.
    load_best_model_at_end=True,
    metric_for_best_model="cer", greater_is_better=False,
    # generation_max_length은 셀 4의 max_new_tokens(72)에 밀린다. 남겨두면
    # 어느 쪽이 먹는지 헷갈리므로 지웠다 — 생성 상한은 셀 4 한 곳에서만 정한다.
    predict_with_generate=True,
    remove_unused_columns=False, label_names=["labels"], report_to="none")
trainer = Seq2SeqTrainer(
    model=model, args=args,
    train_dataset=ds["train"], eval_dataset=ds["dev"],
    data_collator=collator, compute_metrics=compute_metrics,
    processing_class=processor)

# 아래 숫자는 전부 **dev** 기준이다(고르기용). 보고할 성능은 test이고, 그건
# 학습이 끝난 뒤 다음 셀에서 딱 한 번 잰다. 여기 dev CER을 배치 간 비교에
# 인용하지 마라 — 체크포인트를 고르는 데 쓴 값이라 낙관적이다.
# ── baseline 캐시 — 재개할 때마다 다시 재지 않는다 ──────────────────────
# 11차가 327스텝에서 재개했을 때 셀 8이 dev baseline 1536초(25.6분)와 test
# baseline 약 50분을 학습보다 먼저 다시 돌렸다. 값이 정해진 상수인데 세션 시간을
# 그만큼 쓴 것이다(무료 Colab은 세션이 700~850스텝밖에 안 버틴다).
#
# baseline은 **데이터셋(zip)·모델·생성 설정**이 같으면 같다 — 회차와 무관하다.
# 그래서 드라이브에 zip 이름으로 키를 잡아 저장하고, 같은 조건이면 읽는다.
# 12차처럼 같은 데이터셋 위의 다음 회차도 같은 파일을 재사용한다.
BASELINE_PATH = "/content/drive/MyDrive/whisper-608-baseline-" + os.path.basename(ZIP).rsplit(".", 1)[0] + ".json"
_bfp = {"model": MODEL_ID, "n_dev": len(ds["dev"]), "n_test": len(ds["test"]),
        "max_new_tokens": model.generation_config.max_new_tokens,
        "no_repeat_ngram": model.generation_config.no_repeat_ngram_size}
_cached = {}
if os.path.exists(BASELINE_PATH):
    try:
        _c = json.load(open(BASELINE_PATH, encoding="utf-8"))
        if _c.get("fp") == _bfp:
            _cached = _c
        else:
            print("baseline 캐시가 있지만 조건이 다르다 — 다시 잰다:",
                  {k: (_c.get("fp", {}).get(k), v) for k, v in _bfp.items() if _c.get("fp", {}).get(k) != v})
    except Exception as _e:
        print("baseline 캐시를 못 읽었다:", _e)
_cached["fp"] = _bfp


def _save_baseline():
    json.dump(_cached, open(BASELINE_PATH, "w", encoding="utf-8"),
              ensure_ascii=False, indent=1, default=float)


if "dev" in _cached:
    print(f"dev baseline 캐시 사용(재측정 안 함, {BASELINE_PATH}):", _cached["dev"])
elif MEASURE_DEV_BASELINE:
    _dev_base = trainer.evaluate()
    print("파인튜닝 전 baseline (dev):", _dev_base)
    _cached["dev"] = _dev_base
    _save_baseline()
else:
    _base_tag = "big6 7차 실측 0.3101" if RUN not in (FIRST_RUN_BIG7,) else "이 데이터셋 첫 회차인데 생략됨 — 버그"
    print(f"dev baseline 생략 — LoRA B=0 초기화라 회차와 무관한 상수다({_base_tag}).")

# ══ test 계측 헬퍼 — "전"과 "후"를 같은 코드로 잰다 ═══════════════════
# 7차까지 test는 학습이 끝난 뒤에만 쟀다. 그래서 이 프로젝트의 유일한 유효 비교인
# "같은 실행의 전 → 후"가 **dev에만** 있었고 test에는 없었다(7차: dev 0.310 → 0.134,
# test는 후 수치만 존재). 8차부터 test에도 전→후를 만든다.
#
# **학습 전에 test를 보는 것은 오염이 아니다.** 오염은 test 수치로 *선택*을 할 때
# 생긴다(체크포인트·하이퍼파라미터·모델). 학습 전 모델의 숫자는 어떤 선택에도
# 개입하지 않는다. 선택은 여전히 전부 dev로 한다 — 그 원칙은 그대로다.
#
# 시간은 늘지 않는다: 예전 셀 9가 evaluate+predict로 test를 두 번 생성했는데,
# 이제 baseline 1회 + 최종 1회로 같은 두 번이다.
TEST_BASELINE = None   # 학습 전 계측을 담아 두고 셀 9에서 전→후로 찍는다

def report_test(tag):
    """test를 한 번 생성해 앱의 두 지표 + 앱 잣대 통과율을 낸다. dict 반환."""
    import re, unicodedata
    def norm(t):
        t = unicodedata.normalize("NFC", t or "")
        t = re.sub(r"[^\w가-힣\s]", " ", t)
        return re.sub(r"\s+", " ", t).strip()

    pred = trainer.predict(ds["test"])
    lab = pred.label_ids
    lab[lab == -100] = processor.tokenizer.pad_token_id
    hyps = processor.tokenizer.batch_decode(pred.predictions, skip_special_tokens=True)
    refs = processor.tokenizer.batch_decode(lab, skip_special_tokens=True)

    out = {"overall_cer": cer_metric.compute(predictions=hyps, references=refs)}
    print(f"[TEST · {tag}] 전체 CER {out['overall_cer']:.3f}  (참고용 — 판단은 아래 두 지표로)")

    # 앱의 두 기능에 각각 대응하는 두 지표. 섞은 전체 CER로 판단하지 않는다 —
    # CER은 글자 수 가중이라 이 test에선 93%가 문장으로 결정돼 단어가 안 보인다.
    #
    #   단어 정확 일치율  →  QAB 검사(그림 이름대기·단어 따라말하기)
    #   문장 CER          →  대화 모드, 따라말하기·읽기
    #
    # 단어에 CER 대신 정확도를 쓰는 이유: 참조가 평균 1.4자라 조금만 더 뱉어도 CER이
    # 폭발한다(1글자 정답에 5글자면 CER 5.0). "이 음절을 맞게 인식했나"가 앱이 묻는
    # 질문이다.
    if test_task_types is None:
        print("  (이 zip은 task_type 필드가 없는 구 배치 — 두 지표 분리 불가)")
        return out

    w = [i for i, t in enumerate(test_task_types) if t == "wordlist"]
    s = [i for i, t in enumerate(test_task_types) if t == "narrative"]
    if w:
        out["word_acc"] = sum(norm(hyps[i]) == norm(refs[i]) for i in w) / len(w)
        out["word_cer"] = cer_metric.compute(predictions=[hyps[i] for i in w],
                                             references=[refs[i] for i in w])
        print(f"  [검사]   단어 정확도 {out['word_acc']:.1%} · CER {out['word_cer']:.3f}  (n={len(w)})")
    if s:
        out["sent_cer"] = cer_metric.compute(predictions=[hyps[i] for i in s],
                                             references=[refs[i] for i in s])
        print(f"  [대화]   문장 CER {out['sent_cer']:.3f}  (n={len(s)})")
        print("           ※ 대본 낭독이라 자유 발화 대화의 낙관적 대리치다.")

    # 앱 잣대 — 위 CER은 표준 편집거리라 앱 조건보다 비관적이다. 앱은 단어에
    # 음소 유사 가중 거리를 쓰고, 실제 판정은 임계값 통과 여부다.
    # **통과율이 제품에 직결되는 숫자**다(평균이 경계 아래여도 통과율은 100%가 아니다).
    try:
        import app_scorer
    except ImportError:
        print("  (앱 채점기 미로드 — 셀 7.5를 먼저 실행하면 앱 잣대 통과율이 나온다)")
        return out
    print()
    print("  ── 앱 잣대(phoneticEditDistance, 정답 경계 0.34) ──")
    if w:
        r = app_scorer.score_batch([hyps[i] for i in w], [refs[i] for i in w], "word")
        out["word_pass"] = r["pass_rate"]
        print(f"  [검사]   통과율 {r['pass_rate']:.1%} · 평균 오류율 {r['error_rate']:.3f}  (n={r['n']})")
    if s:
        r = app_scorer.score_batch([hyps[i] for i in s], [refs[i] for i in s], "sentence")
        out["sent_pass"] = r["pass_rate"]
        print(f"  [대화]   통과율 {r['pass_rate']:.1%} · 평균 오류율 {r['error_rate']:.3f}  (n={r['n']})")
    return out


# 전 → 후의 "전". 여기서 나온 수치로 아무 선택도 하지 않는다(위 주석 참고).
if "test" in _cached:
    TEST_BASELINE = _cached["test"]
    print("test baseline 캐시 사용(재측정 안 함):", TEST_BASELINE)
elif MEASURE_TEST_BEFORE:
    TEST_BASELINE = report_test("전 · 파인튜닝 전")
    _cached["test"] = TEST_BASELINE
    _save_baseline()
else:
    _tag = "big6 위 — 8차 수치를 대장에서 인용" if RUN not in (FIRST_RUN_BIG7,) else "이 데이터셋 첫 회차인데 생략됨 — 버그"
    print(f"test baseline 생략 — 학습 전 모델은 회차 무관 상수({_tag}).")

# ══ 검문소 — 이전 런의 GPU 잔재 ═══════════════════════════════
# **이건 OOM 예측기가 아니라 잔재 탐지기다.** 둘은 다르다 — 시도 2는 여유가
# 11.33 GiB 있었는데도 OOM으로 죽었다(원인은 배치 4). 여유가 넉넉해도 죽을 수
# 있고, 이 검문소는 그걸 막지 못한다. 막는 건 **다른 실패 하나**다:
#
# 2026-08-25 9차. 출력이 이랬다.
#     학습 시작 전 GPU 여유: 4.48 GiB
#     ...
#     Of the allocated memory 14.05 GiB is allocated by PyTorch,
#     and 361.89 MiB is reserved by PyTorch but unallocated.
#
# reserved-but-unallocated가 362MB뿐 → **단편화가 아니라 참조가 살아 있는 텐서**다.
# 셀 4의 expandable_segments도 아래 empty_cache()도 손댈 수 없다(둘 다 "예약됐지만
# 안 쓰는" 캐시만 건드린다). 범인은 같은 런타임에서 앞서 죽은 시도다 — IPython이
# 실패한 셀의 트레이스백을 sys.last_traceback에 붙들면 그 프레임의 지역 변수
# 전부(모델·활성값·옵티마이저)가 살아남는다. Out[] 히스토리도 같은 일을 한다.
# **gc.collect()로는 안 풀린다**(쓰레기가 아니라 진짜 참조다). 셀 재실행도 소용없다.
# 런타임 재시작이 유일한 해법이다.
#
# **검문소를 체크포인트 처리보다 앞에 둔 게 핵심이다.** 9차에서는 순서가 반대라
# 체크포인트를 먼저 치우고 나서 OOM으로 죽었다. 파괴적인 일을 하기 전에 멈춘다.
import gc, os, sys

# ① 이전 예외 트레이스백과 출력 히스토리가 붙잡은 참조를 끊어 본다.
for _attr in ("last_traceback", "last_value", "last_type"):
    if hasattr(sys, _attr):
        try:
            delattr(sys, _attr)
        except Exception:
            pass
try:
    _ip = get_ipython()
    for _k in ("_", "__", "___", "_i", "_ii", "_iii"):
        _ip.user_ns.pop(_k, None)
    _ip.displayhook.flush()          # Out[] 캐시
except Exception:
    pass
gc.collect(); torch.cuda.empty_cache()

# ② 기대값은 **baseline을 돌았든 안 돌았든 같다.** 한때 "생략하면 모델이 아직
#    CPU에 있으니 약 14 GiB"로 잡았다가 정상 실행을 막을 뻔했다(문턱 12.5에
#    실측 11.38이 걸린다). 틀린 이유: **Trainer.__init__이 이미 모델을 GPU로
#    올린다**(`place_model_on_device` → `_move_model_to_device`). train()이
#    올리는 게 아니다. 그래서 이 print에 닿는 시점엔 어느 회차든 fp32 809M =
#    약 3.2 GiB가 GPU에 있다.
#
#    실측 두 개가 이걸 확인해 준다 — 둘 다 잔재 없는 런타임이다:
#      11.33 GiB  7차재현 시도 2 (baseline 실행)
#      11.38 GiB  9차 2026-08-25 재시도 (baseline 생략)  ← 정상 완주 중이던 값
#
#    9차 첫 시도의 4.48은 여기서 **약 6.9 GiB가 모자랐다.**
_floor = 10.0            # 11.3에서 1.3 GiB 여유. 이보다 올리면 정상 실행을 막는다
_free = torch.cuda.mem_get_info()[0] / 2**30
print(f"학습 시작 전 GPU 여유: {_free:.2f} GiB (문턱 {_floor:.1f} · 정상 약 11.3)")
if _free < _floor:
    print()
    print("=" * 70)
    print("중단한다 — 이 런타임에 이전 실행의 GPU 메모리가 살아 있다.")
    print("체크포인트는 건드리지 않았다. 아래를 하고 이 셀부터 다시 돌려라.")
    print()
    print("  1. [런타임 → 세션 다시 시작]")
    print("     ※ '연결 해제 및 삭제'가 아니다. 재시작은 /content의 데이터를 남긴다.")
    print("  2. 셀 2는 건너뛰고 2.5 → 3 → 4 → 5 → 7.5 → 6 → 7 → 8 순서로 재실행")
    print("  3. 이어붙이려면 아래 RESUME을 True로 (설정이 그대로일 때만)")
    print()
    print("재시작 뒤에도 문턱에 걸리면 그건 잔재가 아니다. 그때는 배치를 더 못 줄인다 —")
    print("이미 배치 2 x 누적 4이고, 체크포인팅은 학습을 망가뜨려 배제됐다")
    print("(시도 3: dev CER 0.8695 vs 0.1700). 대장의 실패 표부터 다시 읽어라.")
    print("=" * 70)
    raise SystemExit("GPU 잔재 — 런타임을 재시작하고 다시 실행해라.")

# ══ 이어붙이기 — 설정 지문으로 자동 판정한다 ═══════════════════
# 이 노트북이 가장 많이 잃은 것은 GPU 시간이고, 잃은 방식은 언제나 같았다:
# **세션이 끊겨 다시 돌리는데 0스텝부터 시작한다.** 9차에서만 세 번 겪었다
# (572·849·729스텝). 체크포인트는 드라이브에 멀쩡히 있었는데도 그랬다.
#
# 원인은 `RESUME` 기본값이 False였던 것이고, 그 기본값에는 이유가 있었다 —
# 7차재현 시도 4에서 자동 이어붙이기가 **망가진 체크포인트를 조용히 물려받았다**
# (체크포인팅으로 학습이 깨진 286스텝. OOM이 우연히 막아줬을 뿐이다).
#
# 두 위험은 사실 하나의 질문으로 갈린다: **체크포인트의 설정이 지금과 같은가?**
#   같다  → 세션이 끊긴 것뿐이다. 이어붙이는 게 항상 옳다.
#   다르다 → 다른 실험의 가중치다. 이어붙이면 안 된다.
#
# 사람의 기억 대신 지문으로 답한다. 학습을 시작할 때 설정을 CKPT_DIR에 적어 두고,
# 다음 실행에서 대조한다. 일치하면 **묻지 않고 이어붙이고**, 어긋나면
# **아무것도 건드리지 않고 멈춘다**(무엇이 달라졌는지 찍어 준다).
import json as _json, hashlib
from transformers.trainer_utils import get_last_checkpoint

# "auto" = 지문으로 판정(권장) | True = 무조건 이어붙임 | False = 처음부터
RESUME = "auto"

# ── 값을 정규화한다 ────────────────────────────────────────────
# "auto"가 문자열이라 True에도 따옴표를 붙이기 쉽다. 그런데 문자열 "True"는
# 불리언 True가 아니라서 아래 `RESUME is True`에 안 걸리고, **조용히 auto처럼
# 동작해 그대로 멈춘다.** 왜 안 되는지 화면에 아무 단서도 안 나온다.
# 실제로 겪은 실수라 받아서 고쳐준다. 이상한 값이면 여기서 바로 말해준다.
if isinstance(RESUME, str):
    _r = RESUME.strip().lower()
    if _r in ("true", "false"):
        print(f'RESUME이 문자열 {RESUME!r}였다 → {_r == "true"}로 읽는다 (따옴표는 빼는 게 맞다).')
        RESUME = (_r == "true")
    elif _r != "auto":
        raise SystemExit(f'RESUME 값이 이상하다: {RESUME!r} — "auto" | True | False 중 하나여야 한다.')
    else:
        RESUME = "auto"
elif RESUME is not True and RESUME is not False:
    raise SystemExit(f'RESUME 값이 이상하다: {RESUME!r} — "auto" | True | False 중 하나여야 한다.')

_fp = {
    "RUN": RUN, "model": MODEL_ID,
    "lora": sorted(LORA_TARGETS), "upsample": WORD_UPSAMPLE,
    "max_steps": MAX_STEPS, "eval_every": EVAL_EVERY,
    "lr": args.learning_rate, "seed": args.seed,
    "bs": args.per_device_train_batch_size,
    "accum": args.gradient_accumulation_steps,
    "eval_bs": args.per_device_eval_batch_size,
    "grad_ckpt": bool(args.gradient_checkpointing),
    "n_train": len(ds["train"]), "n_dev": len(ds["dev"]),
}
_FP_PATH = os.path.join(CKPT_DIR, "run_config.json")
_ckpt = get_last_checkpoint(CKPT_DIR) if os.path.isdir(CKPT_DIR) else None
_resume = None

def _steps_of(p):
    try:
        return int(os.path.basename(p).rsplit("-", 1)[-1])
    except ValueError:
        return None

if _ckpt and RESUME is not False:
    _old = None
    if os.path.exists(_FP_PATH):
        try:
            _old = _json.load(open(_FP_PATH, encoding="utf-8"))
        except Exception as e:
            print("지문을 못 읽었다:", e)
    _same = (_old == _fp)
    if RESUME is True and not _same:
        # 지문이 **없는** 것과 **다른** 것은 다르다. 없는 건 이 기능보다 먼저
        # 만들어진 체크포인트라는 뜻일 뿐이라, 설정이 같다면 결과도 멀쩡하다.
        # 다른 건 진짜로 다른 실험이다. 경고를 뭉뚱그리면 멀쩡한 회차를
        # 버리게 된다.
        if _old is None:
            print("RESUME=True — 지문 없는 체크포인트에 이어붙인다(이 기능 이전에 만들어진 것).")
            print("  설정이 같다면 결과는 유효하다. 이번 실행이 지문을 남기므로 다음부터는 auto로 된다.")
        else:
            print("⚠️ RESUME=True 강제 — 지문이 **다른데** 이어붙인다. 결과를 회차 비교에 쓰지 마라.")
        _same = True
    if _same:
        _resume = _ckpt
    else:
        # **아무것도 옮기지 않는다.** 옮기는 순간 되돌리기가 사람 손에 달린다.
        #
        # 안내는 두 경우를 갈라야 한다. 뭉뚱그렸다가 실제로 틀린 안내를 했다:
        # 지문 없는 572스텝에 "이어붙이면 회차 비교에 못 쓴다"고 찍었는데,
        # 설정이 같은 체크포인트였다. 그 말을 따랐으면 572스텝을 또 버렸을 것이다.
        _lost = _steps_of(_ckpt)
        print("=" * 70)
        if _old is None:
            print("멈춘다 — 체크포인트의 설정을 **확인할 수 없다**(다르다는 뜻이 아니다).")
            print("  체크포인트:", _ckpt, f"({_lost}스텝)" if _lost else "")
            print("  지문이 없다. 이 기능이 생기기 전에 만들어진 체크포인트라서다.")
            print()
            print("이 체크포인트를 만든 실행 이후로 셀 4의 회차/LoRA/업샘플과")
            print("셀 8의 스텝/배치/LR을 **바꾼 적이 없다면** 이어붙이는 게 맞다:")
            print(f"  · RESUME = True   → {_lost}스텝부터 이어감. **결과는 유효하다**"
                  if _lost else "  · RESUME = True   → 이어감. 결과는 유효하다")
            print("  · RESUME = False  → 이 설정으로 0스텝부터(이전 것은 옆으로 치움)")
            print()
            print("이번 실행이 지문을 남기므로 다음부터는 auto가 알아서 판정한다.")
        else:
            print("멈춘다 — 체크포인트가 **다른 설정**의 실행에서 나왔다.")
            print("  체크포인트:", _ckpt, f"({_lost}스텝)" if _lost else "")
            for k in sorted(set(_old) | set(_fp)):
                a, b = _old.get(k, "(없음)"), _fp.get(k, "(없음)")
                if a != b:
                    print(f"    {k}: 체크포인트 {a!r}  →  지금 {b!r}")
            print()
            print("이어붙이면 다른 실험의 가중치를 물려받는다(7차재현 시도 4에서 겪었다).")
            print("  · RESUME = False  → 이 설정으로 0스텝부터  ← 보통 이쪽이다")
            print("  · RESUME = True   → 그래도 이어붙임(회차 비교에는 못 쓴다)")
        print("=" * 70)
        raise SystemExit("체크포인트를 확인할 수 없다 — 위에서 하나를 고르고 다시 실행해라."
                         if _old is None else
                         "체크포인트 설정 불일치 — 위에서 하나를 고르고 다시 실행해라.")

if _ckpt and RESUME is False:
    # 지우지 않고 옆으로 치운다. 지웠는데 사실 이어붙이려던 거였으면 GPU
    # 한 시간이 날아간다. LoRA 체크포인트는 작아서 보관이 싸다.
    import shutil, time
    _aside = f"{CKPT_DIR}-폐기-{time.strftime('%m%d-%H%M')}"
    _lost = _steps_of(_ckpt)
    _cost = f"{_lost}스텝 = 예산 {MAX_STEPS}의 {_lost / MAX_STEPS:.0%}" if _lost else "스텝 수 불명"
    shutil.move(CKPT_DIR, _aside)
    print(f"RESUME=False → 이전 체크포인트를 버리고 0스텝부터 시작한다 ({_cost}).")
    print("  치운 곳:", _aside)
    print("  이어붙이려던 거였다면 **지금 멈추고** 되돌린 뒤 RESUME = \"auto\"로 실행해라:")
    print(f"  !mv '{_aside}' '{CKPT_DIR}'")

if _resume:
    _done = _steps_of(_resume)
    _msg = f"  ({_done} → {MAX_STEPS} 스텝, {MAX_STEPS - _done} 남음)" if _done else ""
    print(f"이어서 학습(지문 일치): {_resume}{_msg}")
else:
    print(f"처음부터 학습한다: {CKPT_DIR}")

# 지문을 남긴다 — 다음 실행이 이걸 보고 이어붙일지 판정한다.
os.makedirs(CKPT_DIR, exist_ok=True)
with open(_FP_PATH, "w", encoding="utf-8") as _f:
    _json.dump(_fp, _f, ensure_ascii=False, indent=1, sort_keys=True)

trainer.train(resume_from_checkpoint=_resume)

# load_best_model_at_end가 최고 체크포인트를 이미 복원했고, 그 dev CER은
# state에 들어 있다. 여기서 trainer.evaluate()를 또 부르면 같은 숫자를 위해
# dev 326개를 다시 생성하는 셈이라 지웠다(셀 9가 state에서 곡선째 찍는다).
print("최고 dev CER:", trainer.state.best_metric, "|", trainer.state.best_model_checkpoint)


## 셀 9 — dev 곡선(판정) + 어댑터 저장. test는 채택 회차에서만

In [ ]:
# ══ 어댑터부터 저장한다 ═══════════════════════════════════════
# 계측보다 이게 먼저다. 여기서 세션이 끊기면 학습 몇 시간이 통째로 날아간다.
# 드라이브에 저장하면 나중에 다른 세션에서 불러 test만 재는 것도 가능하다.
ADAPTER_OUT = f"/content/drive/MyDrive/whisper-608-lora-{RUN}"
model.save_pretrained(ADAPTER_OUT)
print("어댑터 저장:", ADAPTER_OUT)

# ══ dev 곡선 — 탐색 회차의 판정은 전부 여기서 끝난다 ══════════════
# 전체 CER만으로는 9차를 판단할 수 없다. 단어가 세그먼트의 절반이어도 짧아서
# 전체 CER에 묻히기 때문이다. 그래서 셀 7이 단어/문장을 분리해 낸다.
#
#   8차 판정: dev 전체 CER  vs  7차 0.1337   (차이 <0.01이면 좁은 쪽 채택)
#   9차 판정: dev 단어 CER  vs  8차          (문장 CER은 그 대가)
print()
print(f"  ── [{RUN}] dev 곡선 ──────────────────────────────")
print(f"  {'step':>6} {'전체':>8} {'단어':>8} {'문장':>8}")
_nan = float("nan")
for h in trainer.state.log_history:
    if "eval_cer" not in h:
        continue
    print(f"  {int(h.get('step', 0)):6d} {h['eval_cer']:8.4f}"
          f" {h.get('eval_cer_word', _nan):8.4f} {h.get('eval_cer_sent', _nan):8.4f}")
print(f"  최고(전체 CER 기준): {trainer.state.best_metric}")

# ══ test — 채택 회차에서만 ════════════════════════════════════
# 학습·체크포인트 선택은 전부 dev로 끝났다. 이 아래 수치만 홀드아웃 성능이고,
# 모델 채택 판단에 인용할 수 있는 것도 이것뿐이다.
#
# **탐색 회차에서 test를 보지 않는 이유는 시간이 아니라 규율이다.** 회차마다
# test를 보고 나중에 "좋았던 회차"를 고르면, 그건 test로 고른 것이다 — 1~6차를
# 무효로 만든 그 실수와 같다. 승자는 dev로 정하고, 그 다음에 한 번만 본다.
#
# 켜는 법: 위 셀 8의 MEASURE_TEST_AFTER를 True로 바꾸고 셀 8·9를 다시 실행한다.
# 세션이 살아 있으면 재학습 없이 이 셀만 다시 돌려도 된다.
if not MEASURE_TEST_AFTER:
    print()
    print("  test 미측정 (탐색 회차). 승자가 정해지면 MEASURE_TEST_AFTER=True로 잰다.")
else:
    # 계측은 셀 8에서 정의한 report_test로 한다 — baseline과 **같은 코드**여야
    # 전→후가 같은 잣대 위에 놓인다.
    after = report_test("후 · 최고 체크포인트")

    # ── 전 → 후 (같은 test) ────────────────────────────────────
    # 다른 배치의 수치와 가로로 비교하지 마라 — test 오디오가 배치마다 다르다.
    if TEST_BASELINE:
        print()
        print("  ── test 전 → 후 ───────────────────────────────")
        rows = [("전체 CER", "overall_cer", False), ("단어 정확도", "word_acc", True),
                ("단어 CER", "word_cer", False), ("문장 CER", "sent_cer", False),
                ("검사 통과율", "word_pass", True), ("대화 통과율", "sent_pass", True)]
        for label, key, higher_better in rows:
            b, a = TEST_BASELINE.get(key), after.get(key)
            if b is None or a is None:
                continue
            fmt = (lambda v: f"{v:.1%}") if higher_better else (lambda v: f"{v:.3f}")
            delta = (a - b) / b * 100 if b else float("nan")
            arrow = "개선" if ((a > b) == higher_better) else "악화"
            print(f"  {label:10s} {fmt(b)} → {fmt(a)}  ({delta:+.0f}% · {arrow})")
    else:
        print("  (TEST_BASELINE 없음 — 학습 전 test는 8차 수치를 대장에서 인용한다)")

## 셀 10 — 모델 크기 zero-shot 비교 (학습 없음)

**무엇을 정하려는 비교인가**: 파인튜닝 베이스 모델을 small에서 키울지 여부.

판단 기준을 미리 정해 둔다(수치를 보고 정하면 자기합리화가 된다).

| 관측 | 결론 |
|---|---|
| large zero-shot이 파인튜닝한 small보다 좋음 | 베이스 교체가 1순위 |
| large가 **단어에서만** 크게 좋음 | 단어는 용량 문제 → 큰 모델 + 단어 데이터 |
| 모든 모델이 단어에서 비슷하게 나쁨 | 용량이 아니라 도메인(구음장애) 문제 → 데이터가 계속 레버 |
| medium이 large-turbo와 비슷 | 비용상 medium 채택 |


In [ ]:
# ══ 기본으로 건너뛴다 ═══════════════════════════════════════
# 모델 크기 zero-shot 비교(small/medium/turbo)는 이미 끝났고 turbo를 채택했다.
# 전체 실행에 딸려 돌면 test 417개를 모델 3개로 생성해 30분 이상을 먹는다.
RUN_ZERO_SHOT = False
if not RUN_ZERO_SHOT:
    print("zero-shot 비교 건너뜀 — 필요하면 RUN_ZERO_SHOT = True")
    raise SystemExit

# 앱의 두 기능에 각각 대응하는 **두 지표**를 따로 낸다. 섞지 않는다.
#
#   단어 정확 일치율  →  QAB 검사(그림 이름대기·단어 따라말하기)
#   문장 CER          →  대화 모드, 따라말하기·읽기
#
# 왜 섞으면 안 되나: CER은 글자 수로 가중되는 마이크로평균인데 이 test는 단어
# 참조가 210자, 문장이 2807자다. 전체 CER의 93%가 문장으로 결정돼 단어 성능이
# 사실상 안 보인다. 그렇다고 문장이 덜 중요한 것도 아니라(대화 모드), 둘 다
# 1급 지표로 두고 각각 보고한다.
#
# 단어에 CER 대신 정확 일치율을 쓰는 이유: 참조가 평균 1.4자라 조금만 더 뱉어도
# CER이 폭발한다(1글자 정답에 5글자를 뱉으면 CER 5.0). "이 음절을 맞게 인식했나"가
# 앱이 실제로 묻는 질문이고, 정확도 34%가 CER 0.890보다 해석 가능하다.
#
# 단어 클립에는 max_new_tokens를 좁힌다. 0.58초 클립에 100토큰을 허용하면 whisper가
# 문장을 지어내고, 그 환각이 수치를 지배한다. 모델 전부에 동일 적용해야 비교가 산다.

import gc, torch
from collections import Counter
from datasets import load_dataset, Audio
from transformers import WhisperForConditionalGeneration, WhisperProcessor

ZERO_SHOT_MODELS = [
    "openai/whisper-small",            # 현재 베이스
    "openai/whisper-medium",
    "openai/whisper-large-v3-turbo",
]
BATCH = 8
WORD_MAX_TOKENS = 8      # 단어 클립: 환각 억제
SENT_MAX_TOKENS = 100    # 문장 클립: 기존과 동일

raw = load_dataset("json", data_files={"test": f"{BASE}/test.jsonl"})["test"]
raw = raw.map(lambda b: {"audio": os.path.join(BASE, b["audio"])})
raw = raw.cast_column("audio", Audio(sampling_rate=16000))
if "task_type" not in raw.column_names:
    raise SystemExit("이 zip에는 task_type이 없다 — 5차 이후 패키지를 써라.")
task_types = list(raw["task_type"])
refs_all = list(raw["text"])
w_idx = [i for i, t in enumerate(task_types) if t == "wordlist"]
s_idx = [i for i, t in enumerate(task_types) if t == "narrative"]
print(f"test {len(raw)}개 · 단어 {len(w_idx)} · 문장 {len(s_idx)}")

def norm(t):
    """비교용 정규화 — metrics.normalize와 같은 규칙(부호 제거, 공백 축약)."""
    import re, unicodedata
    t = unicodedata.normalize("NFC", t or "")
    t = re.sub(r"[^\w가-힣\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

def transcribe(model_id, idxs, max_new_tokens, batch=BATCH):
    proc = WhisperProcessor.from_pretrained(model_id, language="ko", task="transcribe")
    m = WhisperForConditionalGeneration.from_pretrained(
        model_id, torch_dtype=torch.float16).to("cuda").eval()
    # 생성 설정은 모델 간 완전히 동일하게. 다르면 모델이 아니라 디코딩을 비교하게 된다.
    m.generation_config.language = "ko"
    m.generation_config.task = "transcribe"
    m.generation_config.no_repeat_ngram_size = 3
    m.generation_config.max_new_tokens = max_new_tokens
    m.generation_config.forced_decoder_ids = None

    hyps = []
    for i in range(0, len(idxs), batch):
        chunk = [raw[j]["audio"] for j in idxs[i : i + batch]]
        feats = proc.feature_extractor(
            [a["array"] for a in chunk], sampling_rate=16000, return_tensors="pt"
        ).input_features.to("cuda", torch.float16)
        with torch.no_grad():
            ids = m.generate(feats)
        hyps.extend(proc.tokenizer.batch_decode(ids, skip_special_tokens=True))
    del m, proc
    gc.collect(); torch.cuda.empty_cache()
    return hyps

rows = []
for mid in ZERO_SHOT_MODELS:
    try:
        # 모델마다 자기 프로세서를 로드한다 — large-v3 계열은 mel 128빈, small/medium은
        # 80빈이라 재사용하면 입력이 깨진다(예외 없이 CER만 조용히 나빠진다).
        w_hyp = transcribe(mid, w_idx, WORD_MAX_TOKENS)
        s_hyp = transcribe(mid, s_idx, SENT_MAX_TOKENS)
    except torch.cuda.OutOfMemoryError:
        gc.collect(); torch.cuda.empty_cache()
        print(f"{mid}: OOM → 배치 2로 재시도")
        w_hyp = transcribe(mid, w_idx, WORD_MAX_TOKENS, batch=2)
        s_hyp = transcribe(mid, s_idx, SENT_MAX_TOKENS, batch=2)
    except Exception as e:                     # 한 모델이 죽어도 비교는 계속한다
        print(f"{mid}: 실패 — {type(e).__name__}: {e}")
        continue

    w_ref = [refs_all[i] for i in w_idx]
    s_ref = [refs_all[i] for i in s_idx]
    exact = sum(norm(h) == norm(r) for h, r in zip(w_hyp, w_ref)) / max(len(w_ref), 1)
    rows.append({
        "model": mid.split("/")[-1],
        "단어 정확도": exact,
        "단어 CER": cer_metric.compute(predictions=w_hyp, references=w_ref),
        "문장 CER": cer_metric.compute(predictions=s_hyp, references=s_ref),
    })
    print("  ", {k: (f"{v:.3f}" if isinstance(v, float) else v) for k, v in rows[-1].items()})

print()
print(f"{'model':<24}{'단어 정확도':>12}{'단어 CER':>10}{'문장 CER':>10}")
for r in rows:
    print(f"{r['model']:<24}{r['단어 정확도']:>11.1%}{r['단어 CER']:>10.3f}{r['문장 CER']:>10.3f}")
print()
print("※ 검사 기능은 '단어 정확도', 대화 모드는 '문장 CER'로 본다. 둘을 섞은 값은 쓰지 않는다.")
print("※ 문장은 대본 낭독이라 대화 모드의 낙관적 대리치다 — 자유 발화 평가셋은 별도 과제.")
print("※ zero-shot(학습 없음)이다. 파인튜닝한 모델은 셀 9에서 같은 두 지표로 본다.")
